# Notebook 03: PySpark Ingestion and Fundamentals

## Objective

This notebook introduces the PySpark DataFrame API using the PaySim financial
transaction dataset.

The notebook will:

- verify the Python and Java environment;
- initialize a local Spark session;
- define an explicit source schema;
- ingest the PaySim CSV;
- inspect the Spark DataFrame and its partitions;
- demonstrate transformations and actions;
- perform aggregations using both the DataFrame API and Spark SQL;
- inspect logical and physical execution plans;
- validate Spark results against the Pandas profiling results;
- write and read a small Parquet output.

This notebook focuses on learning Spark fundamentals. The formal Bronze layer
will be implemented in the next notebook.

In [1]:
# Imports
import os
import platform
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

In [2]:
import os
import sys

python_executable = sys.executable

os.environ["PYSPARK_PYTHON"] = python_executable
os.environ["PYSPARK_DRIVER_PYTHON"] = python_executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python executable:", python_executable)
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("SPARK_LOCAL_IP:", os.environ["SPARK_LOCAL_IP"])

Python executable: c:\Projects\paysim-financial-data-pipeline\.venv\Scripts\python.exe
PYSPARK_PYTHON: c:\Projects\paysim-financial-data-pipeline\.venv\Scripts\python.exe
SPARK_LOCAL_IP: 127.0.0.1


In [3]:
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Operating system:", platform.platform())
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python executable: c:\Projects\paysim-financial-data-pipeline\.venv\Scripts\python.exe
Python version: 3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]
Operating system: Windows-11-10.0.26200-SP0
JAVA_HOME: None


In [4]:
import pyspark

print("PySpark version:", pyspark.__version__)

PySpark version: 4.2.0


In [6]:
java_check = os.system("java -version")

print("Java command exit status:", java_check)

Java command exit status: 0


In [7]:
java_check

0

In [8]:
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

JAVA_HOME: None


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PaySimPySparkFundamentals")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", python_executable)
    .config("spark.pyspark.driver.python", python_executable)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [9]:
# # Create Spark Session
# spark = (
#     SparkSession.builder
#     .appName("PaySimPySparkFundamentals")
#     .master("local[*]")
#     .config("spark.sql.shuffle.partitions", "8")
#     .config("spark.sql.session.timeZone", "UTC")
#     .config("spark.driver.memory", "4g")
#     .getOrCreate()
# )

# spark.sparkContext.setLogLevel("WARN")

c:\Projects\paysim-financial-data-pipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)

Spark version: 4.2.0
Spark master: local[4]
Default parallelism: 4


In [6]:
transaction_schema = StructType(
    [
        StructField("step", IntegerType(), nullable=False),
        StructField("type", StringType(), nullable=False),
        StructField("amount", DoubleType(), nullable=False),
        StructField("nameOrig", StringType(), nullable=False),
        StructField("oldbalanceOrg", DoubleType(), nullable=True),
        StructField("newbalanceOrig", DoubleType(), nullable=True),
        StructField("nameDest", StringType(), nullable=False),
        StructField("oldbalanceDest", DoubleType(), nullable=True),
        StructField("newbalanceDest", DoubleType(), nullable=True),
        StructField("isFraud", IntegerType(), nullable=False),
        StructField("isFlaggedFraud", IntegerType(), nullable=False),
    ]
)

print(transaction_schema.simpleString())

struct<step:int,type:string,amount:double,nameOrig:string,oldbalanceOrg:double,newbalanceOrig:double,nameDest:string,oldbalanceDest:double,newbalanceDest:double,isFraud:int,isFlaggedFraud:int>


## Explicit schema decision

The CSV is loaded using a predefined `StructType` rather than
`inferSchema=True`.

Benefits:

- source data types are documented in code;
- unexpected type changes are easier to detect;
- Spark avoids the additional scan used for schema inference;
- repeated pipeline runs apply the same schema consistently;
- downstream transformations receive predictable column types.

The balance columns are nullable because parsing failures or future source
variations may produce nulls, even though the current source profile contains
no missing values.

In [7]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "PS_20174392719_1491204439457_log.csv"
)

REPORT_OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "gold"
    / "data_quality"
)

REPORT_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(RAW_DATA_PATH)
print(REPORT_OUTPUT_PATH)

c:\Projects\paysim-financial-data-pipeline\data\raw\PS_20174392719_1491204439457_log.csv
c:\Projects\paysim-financial-data-pipeline\data\gold\data_quality


In [8]:
transactions_df = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .schema(transaction_schema)
    .csv(str(RAW_DATA_PATH))
)

## Lazy evaluation

The preceding CSV read defines a Spark DataFrame and its computation plan.

Spark transformations are lazy:

- Spark records the requested operations;
- it does not necessarily execute the complete pipeline immediately;
- execution begins when an action is called.

Examples of transformations:

- `select`
- `filter`
- `withColumn`
- `groupBy`
- `orderBy`

Examples of actions:

- `count`
- `show`
- `collect`
- `first`
- writing output files

Lazy evaluation allows Spark to optimize the complete execution plan before
running it.

In [9]:
transactions_df.printSchema()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



In [10]:
# Inspect Columns and types
print("Number of columns:", len(transactions_df.columns))
print("Columns:", transactions_df.columns)

Number of columns: 11
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [11]:
transactions_df.dtypes

[('step', 'int'),
 ('type', 'string'),
 ('amount', 'double'),
 ('nameOrig', 'string'),
 ('oldbalanceOrg', 'double'),
 ('newbalanceOrig', 'double'),
 ('nameDest', 'string'),
 ('oldbalanceDest', 'double'),
 ('newbalanceDest', 'double'),
 ('isFraud', 'int'),
 ('isFlaggedFraud', 'int')]

In [12]:
spark_schema_profile = spark.createDataFrame(
    [
        (
            field.name,
            field.dataType.simpleString(),
            field.nullable,
        )
        for field in transactions_df.schema.fields
    ],
    ["column_name", "spark_data_type", "nullable"],
)

spark_schema_profile.show(
    n=len(transactions_df.columns),
    truncate=False,
)

+--------------+---------------+--------+
|column_name   |spark_data_type|nullable|
+--------------+---------------+--------+
|step          |int            |true    |
|type          |string         |true    |
|amount        |double         |true    |
|nameOrig      |string         |true    |
|oldbalanceOrg |double         |true    |
|newbalanceOrig|double         |true    |
|nameDest      |string         |true    |
|oldbalanceDest|double         |true    |
|newbalanceDest|double         |true    |
|isFraud       |int            |true    |
|isFlaggedFraud|int            |true    |
+--------------+---------------+--------+



In [13]:
source_record_count = transactions_df.count()

print(f"Source record count: {source_record_count:,}")

Source record count: 6,362,620


In [14]:
transactions_df.show(
    n=5,
    truncate=False,
)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|type    |amount  |nameOrig   |oldbalanceOrg|newbalanceOrig|nameDest   |oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|1   |PAYMENT |9839.64 |C1231006815|170136.0     |160296.36     |M1979787155|0.0           |0.0           |0      |0             |
|1   |PAYMENT |1864.28 |C1666544295|21249.0      |19384.72      |M2044282225|0.0           |0.0           |0      |0             |
|1   |TRANSFER|181.0   |C1305486145|181.0        |0.0           |C553264065 |0.0           |0.0           |1      |0             |
|1   |CASH_OUT|181.0   |C840083671 |181.0        |0.0           |C38997010  |21182.0       |0.0           |1      |0             |
|1   |PAYMENT |11668.14|C2048537720|41554.0      |29885.86      |M1230701703|0.0   

In [15]:
transactions_df.select(
    "step",
    "type",
    "amount",
    "nameOrig",
    "nameDest",
    "isFraud",
    "isFlaggedFraud",
).show(
    n=10,
    truncate=False,
)

+----+--------+--------+-----------+-----------+-------+--------------+
|step|type    |amount  |nameOrig   |nameDest   |isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-----------+-------+--------------+
|1   |PAYMENT |9839.64 |C1231006815|M1979787155|0      |0             |
|1   |PAYMENT |1864.28 |C1666544295|M2044282225|0      |0             |
|1   |TRANSFER|181.0   |C1305486145|C553264065 |1      |0             |
|1   |CASH_OUT|181.0   |C840083671 |C38997010  |1      |0             |
|1   |PAYMENT |11668.14|C2048537720|M1230701703|0      |0             |
|1   |PAYMENT |7817.71 |C90045638  |M573487274 |0      |0             |
|1   |PAYMENT |7107.77 |C154988899 |M408069119 |0      |0             |
|1   |PAYMENT |7861.64 |C1912850431|M633326333 |0      |0             |
|1   |PAYMENT |4024.36 |C1265012928|M1176932104|0      |0             |
|1   |DEBIT   |5337.77 |C712410124 |C195600860 |0      |0             |
+----+--------+--------+-----------+-----------+-------+--------

In [16]:
first_transaction = transactions_df.first()

first_transaction

Row(step=1, type='PAYMENT', amount=9839.64, nameOrig='C1231006815', oldbalanceOrg=170136.0, newbalanceOrig=160296.36, nameDest='M1979787155', oldbalanceDest=0.0, newbalanceDest=0.0, isFraud=0, isFlaggedFraud=0)

In [17]:
print("Step:", first_transaction["step"])
print("Type:", first_transaction["type"])
print("Amount:", first_transaction["amount"])

Step: 1
Type: PAYMENT
Amount: 9839.64


In [18]:
selected_transactions_df = transactions_df.select(
    "step",
    "type",
    "amount",
    "nameOrig",
    "nameDest",
    "isFraud",
)

selected_transactions_df.show(5, truncate=False)

+----+--------+--------+-----------+-----------+-------+
|step|type    |amount  |nameOrig   |nameDest   |isFraud|
+----+--------+--------+-----------+-----------+-------+
|1   |PAYMENT |9839.64 |C1231006815|M1979787155|0      |
|1   |PAYMENT |1864.28 |C1666544295|M2044282225|0      |
|1   |TRANSFER|181.0   |C1305486145|C553264065 |1      |
|1   |CASH_OUT|181.0   |C840083671 |C38997010  |1      |
|1   |PAYMENT |11668.14|C2048537720|M1230701703|0      |
+----+--------+--------+-----------+-----------+-------+
only showing top 5 rows


In [19]:
fraud_transactions_df = transactions_df.filter(
    F.col("isFraud") == 1
)

fraud_count = fraud_transactions_df.count()

print(f"Fraudulent transactions: {fraud_count:,}")

Fraudulent transactions: 8,213


In [20]:
transactions_df[
    transactions_df["isFraud"] == 1
].select(
    "step",
    "type",
    "amount",
    "nameOrig",
    "nameDest",
).show(10, truncate=False)

+----+--------+----------+-----------+-----------+
|step|type    |amount    |nameOrig   |nameDest   |
+----+--------+----------+-----------+-----------+
|1   |TRANSFER|181.0     |C1305486145|C553264065 |
|1   |CASH_OUT|181.0     |C840083671 |C38997010  |
|1   |TRANSFER|2806.0    |C1420196421|C972765878 |
|1   |CASH_OUT|2806.0    |C2101527076|C1007251739|
|1   |TRANSFER|20128.0   |C137533655 |C1848415041|
|1   |CASH_OUT|20128.0   |C1118430673|C339924917 |
|1   |CASH_OUT|416001.33 |C749981943 |C667346055 |
|1   |TRANSFER|1277212.77|C1334405552|C431687661 |
|1   |CASH_OUT|1277212.77|C467632528 |C716083600 |
|1   |TRANSFER|35063.63  |C1364127192|C1136419747|
+----+--------+----------+-----------+-----------+
only showing top 10 rows


In [21]:
high_value_fraud_df = transactions_df.filter(
    (F.col("isFraud") == 1)
    & (F.col("amount") > 200_000)
)

print(
    "High-value fraudulent transactions:",
    f"{high_value_fraud_df.count():,}",
)

High-value fraudulent transactions: 5,471


In [22]:
high_value_fraud_df.select(
    "step",
    "type",
    "amount",
    "nameOrig",
    "nameDest",
    "isFlaggedFraud",
).show(10, truncate=False)

+----+--------+----------+-----------+-----------+--------------+
|step|type    |amount    |nameOrig   |nameDest   |isFlaggedFraud|
+----+--------+----------+-----------+-----------+--------------+
|1   |CASH_OUT|416001.33 |C749981943 |C667346055 |0             |
|1   |TRANSFER|1277212.77|C1334405552|C431687661 |0             |
|1   |CASH_OUT|1277212.77|C467632528 |C716083600 |0             |
|1   |TRANSFER|235238.66 |C1872047468|C116289363 |0             |
|1   |CASH_OUT|235238.66 |C1499825229|C2100440237|0             |
|2   |TRANSFER|1096187.24|C1093223281|C2063275841|0             |
|2   |CASH_OUT|1096187.24|C77163673  |C644345897 |0             |
|2   |TRANSFER|963532.14 |C1440057381|C268086000 |0             |
|2   |CASH_OUT|963532.14 |C430329518 |C991505714 |0             |
|4   |TRANSFER|1.0E7     |C7162498   |C945327594 |0             |
+----+--------+----------+-----------+-----------+--------------+
only showing top 10 rows


In [23]:
#Derive Columns with withColumn

enriched_transactions_df = (
    transactions_df
    .withColumn(
        "transaction_day",
        F.ceil(F.col("step") / F.lit(24)).cast("integer"),
    )
    .withColumn(
        "hour_of_day",
        ((F.col("step") - F.lit(1)) % F.lit(24)).cast("integer"),
    )
    .withColumn(
        "destination_category",
        F.when(
            F.col("nameDest").startswith("M"),
            F.lit("MERCHANT"),
        ).otherwise(F.lit("CUSTOMER")),
    )
    .withColumn(
        "is_high_value",
        (F.col("amount") > F.lit(200_000)).cast("integer"),
    )
)

In [24]:
enriched_transactions_df.select(
    "step",
    "transaction_day",
    "hour_of_day",
    "type",
    "amount",
    "nameDest",
    "destination_category",
    "is_high_value",
).show(15, truncate=False)

+----+---------------+-----------+--------+--------+-----------+--------------------+-------------+
|step|transaction_day|hour_of_day|type    |amount  |nameDest   |destination_category|is_high_value|
+----+---------------+-----------+--------+--------+-----------+--------------------+-------------+
|1   |1              |0          |PAYMENT |9839.64 |M1979787155|MERCHANT            |0            |
|1   |1              |0          |PAYMENT |1864.28 |M2044282225|MERCHANT            |0            |
|1   |1              |0          |TRANSFER|181.0   |C553264065 |CUSTOMER            |0            |
|1   |1              |0          |CASH_OUT|181.0   |C38997010  |CUSTOMER            |0            |
|1   |1              |0          |PAYMENT |11668.14|M1230701703|MERCHANT            |0            |
|1   |1              |0          |PAYMENT |7817.71 |M573487274 |MERCHANT            |0            |
|1   |1              |0          |PAYMENT |7107.77 |M408069119 |MERCHANT            |0            |


In [25]:
enriched_transactions_df.select(
    F.min("transaction_day").alias("minimum_day"),
    F.max("transaction_day").alias("maximum_day"),
    F.min("hour_of_day").alias("minimum_hour"),
    F.max("hour_of_day").alias("maximum_hour"),
).show()

+-----------+-----------+------------+------------+
|minimum_day|maximum_day|minimum_hour|maximum_hour|
+-----------+-----------+------------+------------+
|          1|         31|           0|          23|
+-----------+-----------+------------+------------+



In [26]:
#Conditional Column Logic

categorized_transactions_df = (
    enriched_transactions_df
    .withColumn(
        "amount_category",
        F.when(F.col("amount") == 0, "ZERO")
        .when(F.col("amount") < 1_000, "LOW")
        .when(F.col("amount") < 50_000, "MEDIUM")
        .when(F.col("amount") < 200_000, "HIGH")
        .otherwise("VERY_HIGH"),
    )
)

In [27]:
categorized_transactions_df.groupBy(
    "amount_category"
).count().orderBy(
    F.desc("count")
).show()

+---------------+-------+
|amount_category|  count|
+---------------+-------+
|         MEDIUM|2663305|
|           HIGH|1883103|
|      VERY_HIGH|1673570|
|            LOW| 142626|
|           ZERO|     16|
+---------------+-------+



In [28]:
# Transaction Type Counts

transaction_type_counts_df = (
    transactions_df
    .groupBy("type")
    .agg(
        F.count("*").alias("transaction_count"),
    )
    .orderBy(
        F.desc("transaction_count")
    )
)

transaction_type_counts_df.show(truncate=False)

+--------+-----------------+
|type    |transaction_count|
+--------+-----------------+
|CASH_OUT|2237500          |
|PAYMENT |2151495          |
|CASH_IN |1399284          |
|TRANSFER|532909           |
|DEBIT   |41432            |
+--------+-----------------+



In [29]:
#Transaction type percentages

transaction_type_distribution_df = (
    transaction_type_counts_df
    .withColumn(
        "transaction_pct",
        F.round(
            F.col("transaction_count")
            / F.lit(source_record_count)
            * F.lit(100),
            4,
        ),
    )
)

transaction_type_distribution_df.show(truncate=False)

+--------+-----------------+---------------+
|type    |transaction_count|transaction_pct|
+--------+-----------------+---------------+
|CASH_OUT|2237500          |35.1663        |
|PAYMENT |2151495          |33.8146        |
|CASH_IN |1399284          |21.9923        |
|TRANSFER|532909           |8.3756         |
|DEBIT   |41432            |0.6512         |
+--------+-----------------+---------------+



In [30]:
# Fraud distribution

fraud_distribution_df = (
    transactions_df
    .groupBy("isFraud")
    .agg(
        F.count("*").alias("transaction_count"),
    )
    .withColumn(
        "transaction_pct",
        F.round(
            F.col("transaction_count")
            / F.lit(source_record_count)
            * F.lit(100),
            6,
        ),
    )
    .orderBy("isFraud")
)

fraud_distribution_df.show(truncate=False)

+-------+-----------------+---------------+
|isFraud|transaction_count|transaction_pct|
+-------+-----------------+---------------+
|0      |6354407          |99.870918      |
|1      |8213             |0.129082       |
+-------+-----------------+---------------+



In [31]:
# Aggregation by transaction type

transaction_type_summary_df = (
    transactions_df
    .groupBy("type")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(
            F.sum("amount"),
            2,
        ).alias("total_amount"),
        F.round(
            F.avg("amount"),
            2,
        ).alias("average_amount"),
        F.round(
            F.min("amount"),
            2,
        ).alias("minimum_amount"),
        F.round(
            F.max("amount"),
            2,
        ).alias("maximum_amount"),
        F.sum("isFraud").alias("fraud_count"),
        F.sum("isFlaggedFraud").alias("flagged_fraud_count"),
    )
    .withColumn(
        "fraud_rate_pct",
        F.round(
            F.col("fraud_count")
            / F.col("transaction_count")
            * F.lit(100),
            6,
        ),
    )
    .orderBy(
        F.desc("transaction_count")
    )
)

transaction_type_summary_df.show(
    n=10,
    truncate=False,
)

+--------+-----------------+------------------+--------------+--------------+--------------+-----------+-------------------+--------------+
|type    |transaction_count|total_amount      |average_amount|minimum_amount|maximum_amount|fraud_count|flagged_fraud_count|fraud_rate_pct|
+--------+-----------------+------------------+--------------+--------------+--------------+-----------+-------------------+--------------+
|CASH_OUT|2237500          |3.9441299522449E11|176273.96     |0.0           |1.0E7         |4116       |0                  |0.183955      |
|PAYMENT |2151495          |2.809337113837E10 |13057.6       |0.02          |238637.98     |0          |0                  |0.0           |
|CASH_IN |1399284          |2.3636739191246E11|168920.24     |0.04          |1915267.9     |0          |0                  |0.0           |
|TRANSFER|532909           |4.8529198726317E11|910647.01     |2.6           |9.244551664E7 |4097       |16                 |0.768799      |
|DEBIT   |41432     

In [32]:
# Distinct Values

transactions_df.select("type").distinct().show()

+--------+
|    type|
+--------+
| PAYMENT|
|   DEBIT|
| CASH_IN|
|TRANSFER|
|CASH_OUT|
+--------+



In [33]:
transactions_df.select(
    F.countDistinct("step").alias("unique_steps"),
    F.countDistinct("type").alias("unique_transaction_types"),
    F.countDistinct("nameOrig").alias("unique_origin_accounts"),
    F.countDistinct("nameDest").alias("unique_destination_accounts"),
).show()

+------------+------------------------+----------------------+---------------------------+
|unique_steps|unique_transaction_types|unique_origin_accounts|unique_destination_accounts|
+------------+------------------------+----------------------+---------------------------+
|         743|                       5|               6353307|                    2722362|
+------------+------------------------+----------------------+---------------------------+



In [34]:
#Missing Value Profile

null_count_expressions = [
    F.sum(
        F.when(
            F.col(column).isNull(),
            F.lit(1),
        ).otherwise(F.lit(0))
    ).alias(column)
    for column in transactions_df.columns
]

null_counts_row = (
    transactions_df
    .select(*null_count_expressions)
    .first()
)

spark_null_profile = spark.createDataFrame(
    [
        (
            column,
            int(null_counts_row[column]),
        )
        for column in transactions_df.columns
    ],
    ["column_name", "null_count"],
)

spark_null_profile.show(
    n=len(transactions_df.columns),
    truncate=False,
)

+--------------+----------+
|column_name   |null_count|
+--------------+----------+
|step          |0         |
|type          |0         |
|amount        |0         |
|nameOrig      |0         |
|oldbalanceOrg |0         |
|newbalanceOrig|0         |
|nameDest      |0         |
|oldbalanceDest|0         |
|newbalanceDest|0         |
|isFraud       |0         |
|isFlaggedFraud|0         |
+--------------+----------+



In [35]:
spark_null_profile = (
    spark_null_profile
    .withColumn(
        "null_pct",
        F.round(
            F.col("null_count")
            / F.lit(source_record_count)
            * F.lit(100),
            6,
        ),
    )
)

spark_null_profile.show(
    n=len(transactions_df.columns),
    truncate=False,
)

+--------------+----------+--------+
|column_name   |null_count|null_pct|
+--------------+----------+--------+
|step          |0         |0.0     |
|type          |0         |0.0     |
|amount        |0         |0.0     |
|nameOrig      |0         |0.0     |
|oldbalanceOrg |0         |0.0     |
|newbalanceOrig|0         |0.0     |
|nameDest      |0         |0.0     |
|oldbalanceDest|0         |0.0     |
|newbalanceDest|0         |0.0     |
|isFraud       |0         |0.0     |
|isFlaggedFraud|0         |0.0     |
+--------------+----------+--------+



In [36]:
# Check Valid transaction types

valid_transaction_types = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER",
]

invalid_transaction_type_df = transactions_df.filter(
    ~F.col("type").isin(valid_transaction_types)
    | F.col("type").isNull()
)

invalid_transaction_type_count = (
    invalid_transaction_type_df.count()
)

print(
    "Invalid transaction-type records:",
    f"{invalid_transaction_type_count:,}",
)

Invalid transaction-type records: 0


In [37]:
# Check numeric values

numeric_validation_df = transactions_df.select(
    F.sum(
        F.when(F.col("amount") < 0, 1).otherwise(0)
    ).alias("negative_amount_count"),
    F.sum(
        F.when(F.col("amount") == 0, 1).otherwise(0)
    ).alias("zero_amount_count"),
    F.sum(
        F.when(~F.col("isFraud").isin(0, 1), 1).otherwise(0)
    ).alias("invalid_fraud_flag_count"),
    F.sum(
        F.when(
            ~F.col("isFlaggedFraud").isin(0, 1),
            1,
        ).otherwise(0)
    ).alias("invalid_flagged_fraud_count"),
)

numeric_validation_df.show(truncate=False)

+---------------------+-----------------+------------------------+---------------------------+
|negative_amount_count|zero_amount_count|invalid_fraud_flag_count|invalid_flagged_fraud_count|
+---------------------+-----------------+------------------------+---------------------------+
|0                    |16               |0                       |0                          |
+---------------------+-----------------+------------------------+---------------------------+



In [38]:
# Describe numeric values

transactions_df.select(
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
).describe().show(
    truncate=False
)

+-------+------------------+------------------+-----------------+-----------------+------------------+------------------+
|summary|step              |amount            |oldbalanceOrg    |newbalanceOrig   |oldbalanceDest    |newbalanceDest    |
+-------+------------------+------------------+-----------------+-----------------+------------------+------------------+
|count  |6362620           |6362620           |6362620          |6362620          |6362620           |6362620           |
|mean   |243.39724563151657|179861.90354913412|833883.1040744719|855113.6685785714|1100701.6665196654|1224996.3982019408|
|stddev |142.33197104912588|603858.2314629498 |2888242.673037545|2924048.502954253|3399180.1129944855|3674128.9421195714|
|min    |1                 |0.0               |0.0              |0.0              |0.0               |0.0               |
|max    |743               |9.244551664E7     |5.958504037E7    |4.958504037E7    |3.5601588935E8    |3.5617927892E8    |
+-------+---------------

In [39]:
amount_quantiles = transactions_df.approxQuantile(
    "amount",
    [0.01, 0.25, 0.50, 0.75, 0.95, 0.99],
    0.001,
)

amount_quantile_labels = [
    "1%",
    "25%",
    "50%",
    "75%",
    "95%",
    "99%",
]

for label, value in zip(
    amount_quantile_labels,
    amount_quantiles,
    strict=True,
):
    print(f"{label}: {value:,.2f}")

1%: 417.80
25%: 13,387.92
50%: 74,687.88
75%: 208,272.94
95%: 515,787.54
99%: 1,581,193.46


## SPARK SQL

In [40]:
transactions_df.createOrReplaceTempView(
    "paysim_transactions"
)

In [41]:
spark.sql(
    """
    SELECT
        type AS transaction_type,
        COUNT(*) AS transaction_count,
        ROUND(SUM(amount), 2) AS total_amount,
        ROUND(AVG(amount), 2) AS average_amount,
        SUM(isFraud) AS fraud_count,
        ROUND(
            100.0 * SUM(isFraud) / COUNT(*),
            6
        ) AS fraud_rate_pct
    FROM paysim_transactions
    GROUP BY type
    ORDER BY transaction_count DESC
    """
).show(
    truncate=False
)

+----------------+-----------------+------------------+--------------+-----------+--------------+
|transaction_type|transaction_count|total_amount      |average_amount|fraud_count|fraud_rate_pct|
+----------------+-----------------+------------------+--------------+-----------+--------------+
|CASH_OUT        |2237500          |3.9441299522449E11|176273.96     |4116       |0.183955      |
|PAYMENT         |2151495          |2.809337113837E10 |13057.6       |0          |0.000000      |
|CASH_IN         |1399284          |2.3636739191246E11|168920.24     |0          |0.000000      |
|TRANSFER        |532909           |4.8529198726317E11|910647.01     |4097       |0.768799      |
|DEBIT           |41432            |2.2719922128E8    |5483.67       |0          |0.000000      |
+----------------+-----------------+------------------+--------------+-----------+--------------+



In [42]:
# Query Fradulent Transactions

spark.sql(
    """
    SELECT
        step,
        type,
        amount,
        nameOrig,
        nameDest,
        isFlaggedFraud
    FROM paysim_transactions
    WHERE isFraud = 1
    ORDER BY amount DESC
    LIMIT 10
    """
).show(
    truncate=False
)

+----+--------+------+-----------+-----------+--------------+
|step|type    |amount|nameOrig   |nameDest   |isFlaggedFraud|
+----+--------+------+-----------+-----------+--------------+
|362 |TRANSFER|1.0E7 |C1208192074|C255905586 |0             |
|387 |CASH_OUT|1.0E7 |C618976547 |C1908782637|0             |
|362 |CASH_OUT|1.0E7 |C23198921  |C486186579 |0             |
|370 |TRANSFER|1.0E7 |C1802427135|C1072055067|0             |
|370 |CASH_OUT|1.0E7 |C1751546135|C296834383 |0             |
|354 |TRANSFER|1.0E7 |C501435638 |C201807091 |0             |
|386 |TRANSFER|1.0E7 |C1499124218|C430952051 |0             |
|357 |TRANSFER|1.0E7 |C393177637 |C106531078 |0             |
|386 |CASH_OUT|1.0E7 |C634268681 |C879312587 |0             |
|359 |TRANSFER|1.0E7 |C1889901787|C1023006881|0             |
+----+--------+------+-----------+-----------+--------------+



In [43]:
# Hourly Summary 

hourly_summary_df = spark.sql(
    """
    SELECT
        step,
        COUNT(*) AS transaction_count,
        ROUND(SUM(amount), 2) AS total_amount,
        SUM(isFraud) AS fraud_count,
        ROUND(
            100.0 * SUM(isFraud) / COUNT(*),
            6
        ) AS fraud_rate_pct
    FROM paysim_transactions
    GROUP BY step
    ORDER BY step
    """
)

hourly_summary_df.show(10, truncate=False)

+----+-----------------+---------------+-----------+--------------+
|step|transaction_count|total_amount   |fraud_count|fraud_rate_pct|
+----+-----------------+---------------+-----------+--------------+
|1   |2708             |2.8542918115E8 |16         |0.590842      |
|2   |1014             |8.592160402E7  |8          |0.788955      |
|3   |552              |4.329388442E7  |4          |0.724638      |
|4   |565              |7.291002857E7  |10         |1.769912      |
|5   |665              |4.554808975E7  |6          |0.902256      |
|6   |1660             |1.6431055122E8 |22         |1.325301      |
|7   |6837             |8.3293081424E8 |12         |0.175516      |
|8   |21097            |3.43960240735E9|12         |0.056880      |
|9   |37628            |7.00837923943E9|19         |0.050494      |
|10  |35991            |7.12421489371E9|11         |0.030563      |
+----+-----------------+---------------+-----------+--------------+
only showing top 10 rows


In [44]:
# Partitions and Execution Plans

input_partition_count = (
    transactions_df.rdd.getNumPartitions()
)

print("Input partitions:", input_partition_count)

Input partitions: 4


In [45]:
partition_size_df = spark.createDataFrame(
    transactions_df.rdd.mapPartitionsWithIndex(
        lambda partition_index, rows: [
            (
                partition_index,
                sum(1 for _ in rows),
            )
        ]
    ),
    ["partition_id", "record_count"],
)

partition_size_df.orderBy("partition_id").show(
    n=input_partition_count,
    truncate=False,
)

+------------+------------+
|partition_id|record_count|
+------------+------------+
|0           |1612941     |
|1           |1601179     |
|2           |1601794     |
|3           |1546706     |
+------------+------------+



In [46]:
partition_size_df.select(
    F.count("*").alias("partition_count"),
    F.min("record_count").alias("minimum_records"),
    F.max("record_count").alias("maximum_records"),
    F.round(
        F.avg("record_count"),
        2,
    ).alias("average_records"),
).show()

+---------------+---------------+---------------+---------------+
|partition_count|minimum_records|maximum_records|average_records|
+---------------+---------------+---------------+---------------+
|              4|        1546706|        1612941|      1590655.0|
+---------------+---------------+---------------+---------------+



## Repartition and coalesce

`repartition(n)`:

- can increase or decrease the number of partitions;
- usually performs a full shuffle;
- can improve distribution before expensive operations or writes.

`coalesce(n)`:

- is primarily used to reduce partitions;
- generally avoids a full shuffle;
- can produce uneven partitions.

Neither method should be applied without a reason. Excessive repartitioning
creates unnecessary network and disk work.

In [47]:
fraud_partition_demo_df = (
    fraud_transactions_df
    .repartition(4)
)

print(
    "Fraud partitions after repartition:",
    fraud_partition_demo_df.rdd.getNumPartitions(),
)

Fraud partitions after repartition: 4


In [48]:
transaction_type_summary_df.explain(
    mode="formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Sort (7)
   +- Exchange (6)
      +- Project (5)
         +- HashAggregate (4)
            +- Exchange (3)
               +- HashAggregate (2)
                  +- Scan csv  (1)


(1) Scan csv 
Output [4]: [type#1, amount#2, isFraud#9, isFlaggedFraud#10]
Batched: false
Location: InMemoryFileIndex [file:/c:/Projects/paysim-financial-data-pipeline/data/raw/PS_20174392719_1491204439457_log.csv]
ReadSchema: struct<type:string,amount:double,isFraud:int,isFlaggedFraud:int>

(2) HashAggregate
Input [4]: [type#1, amount#2, isFraud#9, isFlaggedFraud#10]
Keys [1]: [type#1]
Functions [7]: [partial_count(1), partial_sum(amount#2), partial_avg(amount#2), partial_min(amount#2), partial_max(amount#2), partial_sum(isFraud#9), partial_sum(isFlaggedFraud#10)]
Aggregate Attributes [8]: [count#428L, sum#429, sum#430, count#431L, min#432, max#433, sum#434L, sum#435L]
Results [9]: [type#1, count#436L, sum#437, sum#438, count#439L, min#440, max#441, sum#442L, sum#

## Execution plan interpretation

Typical operators include:

- **Scan**: reads the CSV source.
- **Project**: selects or derives columns.
- **Filter**: removes rows not satisfying a condition.
- **HashAggregate**: performs grouped aggregation.
- **Exchange**: redistributes data between partitions, usually because of a
  shuffle.
- **Sort**: orders output rows.

An `Exchange` is important because shuffle operations are generally more
expensive than narrow transformations.

In [49]:
transfer_transactions_df = (
    transactions_df
    .filter(F.col("type") == "TRANSFER")
    .select(
        "step",
        "amount",
        "nameOrig",
        "nameDest",
        "isFraud",
    )
)

In [50]:
transfer_transactions_df.explain(
    mode="simple"
)

== Physical Plan ==
*(1) Project [step#0, amount#2, nameOrig#3, nameDest#6, isFraud#9]
+- *(1) Filter (isnotnull(type#1) AND (type#1 = TRANSFER))
   +- FileScan csv [step#0,type#1,amount#2,nameOrig#3,nameDest#6,isFraud#9] Batched: false, DataFilters: [isnotnull(type#1), (type#1 = TRANSFER)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Projects/paysim-financial-data-pipeline/data/raw/PS_201743927..., PartitionFilters: [], PushedFilters: [IsNotNull(type), EqualTo(type,TRANSFER)], ReadSchema: struct<step:int,type:string,amount:double,nameOrig:string,nameDest:string,isFraud:int>




In [51]:
transfer_count = transfer_transactions_df.count()

print(f"Transfer transactions: {transfer_count:,}")

Transfer transactions: 532,909


In [52]:
# Cache Demonstration
fraud_transactions_df.cache()


DataFrame[step: int, type: string, amount: double, nameOrig: string, oldbalanceOrg: double, newbalanceOrig: double, nameDest: string, oldbalanceDest: double, newbalanceDest: double, isFraud: int, isFlaggedFraud: int]

In [53]:
cached_fraud_count = fraud_transactions_df.count()

print(f"Cached fraud records: {cached_fraud_count:,}")

Cached fraud records: 8,213


In [54]:
print(
    "Fraud DataFrame storage level:",
    fraud_transactions_df.storageLevel,
)

Fraud DataFrame storage level: Disk Memory Deserialized 1x Replicated


In [55]:
fraud_transactions_df.groupBy("type").agg(
    F.count("*").alias("fraud_count"),
    F.round(
        F.sum("amount"),
        2,
    ).alias("fraud_amount"),
).orderBy(
    F.desc("fraud_count")
).show()

+--------+-----------+---------------+
|    type|fraud_count|   fraud_amount|
+--------+-----------+---------------+
|CASH_OUT|       4116|5.98920224383E9|
|TRANSFER|       4097|6.06721318401E9|
+--------+-----------+---------------+



In [56]:
fraud_transactions_df.unpersist()

DataFrame[step: int, type: string, amount: double, nameOrig: string, oldbalanceOrg: double, newbalanceOrig: double, nameDest: string, oldbalanceDest: double, newbalanceDest: double, isFraud: int, isFlaggedFraud: int]

## Note

Parquet write validation was intentionally deferred because native Windows Spark
requires Hadoop Windows binaries (winutils.exe and hadoop.dll).

The complete Bronze layer implementation in Notebook 04 will be executed in a
Linux/WSL environment, which more closely reflects production Spark deployments.